# RadioML 2018 training smoke run
This notebook attaches `pinxau1000/radioml2018`, installs RadioFry, verifies the GPU, and runs a bounded 100k-sample / 5-epoch smoke training before any longer experiment.

In [ ]:
from pathlib import Path
import subprocess
import sys

dataset_files = sorted(Path('/kaggle/input').rglob('*.hdf5'))
if not dataset_files:
    raise FileNotFoundError('No HDF5 dataset found under /kaggle/input')
dataset_path = dataset_files[0]
print(f'Dataset: {dataset_path}')
print(f'Size: {dataset_path.stat().st_size / 1024**3:.2f} GiB')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select a GPU accelerator in Kaggle Notebook options.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
repo_dir = Path('/kaggle/working/RadioFry')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/The-4Script/RadioFry.git', str(repo_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir / '.[ml,training]')], check=True)
print(f'RadioFry installed from {repo_dir}')

In [ ]:
output_path = Path('/kaggle/working/modulation_cnn_radioml2018_smoke.pt')
command = [
    sys.executable, '-m', 'radiofry.training.train_modulation',
    '--data', str(dataset_path),
    '--output', str(output_path),
    '--max-samples', '100000',
    '--epochs', '5',
    '--batch-size', '256',
    '--sample-length', '1024',
    '--device', 'cuda',
    '--features', 'iq',
]
print('Starting smoke training:', ' '.join(command))
subprocess.run(command, cwd=repo_dir, check=True)
print(f'Checkpoint: {output_path} ({output_path.stat().st_size / 1024**2:.1f} MiB)')